# 01 · Retrieval: BM25 vs Dense vs Hybrid vs Reranker

**Trước khi chạy** (khung Settings bên phải):
1. **Accelerator:** GPU T4 x2 (hoặc T4 x1)
2. **Internet:** On (tài khoản cần xác minh số điện thoại)

Sau đó bấm **Run All**. Mất khoảng 40–60 phút, chủ yếu để encode khoảng 106K chunk bằng bge-m3.

In [ ]:
REPO_URL = "https://github.com/sinhtruc24/Vietnamese-Legal-RAG.git"
WORK = "/kaggle/working/Vietnamese-Legal-RAG"

import os, socket
try:
    socket.create_connection(("github.com", 443), timeout=5).close()
except OSError:
    raise SystemExit("Không có Internet: Settings (bên phải) -> Internet = On, "
                     "rồi Run -> Restart & clear cell outputs và chạy lại.")

if not os.path.exists(WORK):
    !git clone -q {REPO_URL} {WORK}
else:
    !git -C {WORK} pull -q
assert os.path.exists(WORK), "Clone thất bại: kiểm tra REPO_URL và repo có để Public không"
%cd {WORK}
!git log --oneline -1

In [ ]:
# Kaggle đã có sẵn torch + CUDA; chỉ cài thêm phần còn thiếu
!pip install -q -e ".[ml]" openai huggingface_hub
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Tải dữ liệu Zalo AI 2021 Legal

In [ ]:
!python scripts/download_data.py

## 2. Build index: BM25 + bigram và dense bge-m3
Nếu bị hết bộ nhớ GPU (OOM), giảm `--batch-size` xuống 32 hoặc 16.

In [ ]:
!python scripts/build_index.py --bigrams --index-dir indexes/full --batch-size 64

## 3. Đánh giá retrieval trên 788 câu hỏi test

In [ ]:
!python scripts/eval_retrieval.py --index-dir indexes/full \
    --methods bm25 dense hybrid hybrid+rerank dense+rerank \
    --output results/retrieval_test.json

In [ ]:
import json, pandas as pd
res = json.load(open("results/retrieval_test.json"))["results"]
pd.DataFrame(res).T.round(4)

## 4. Ablation: số ứng viên đưa vào reranker
Reranker chính xác hơn nhưng chậm hơn. Mục này tìm điểm cân bằng giữa chất lượng và độ trễ.

In [ ]:
for n in [10, 20, 50]:
    !python scripts/eval_retrieval.py --index-dir indexes/full --methods hybrid+rerank \
        --rerank-candidates {n} --output results/retrieval_rerank{n}.json

## 5. Lưu kết quả
- Bấm **Save Version → Save & Run All (Commit)** để lưu thư mục `/kaggle/working` (gồm index và kết quả).
- Mở tab **Output** của version đã lưu và tải `results/*.json` về máy để điền vào README.
- Muốn dùng lại index ở notebook khác: trong notebook đó chọn **Add Input → Notebook Output**, rồi chọn notebook này.